In [10]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath("../../"))

In [11]:
import os
import cv2

from backend.services.face_service import get_face_embedding
from backend.services.validation_service import calculate_similarity

DATASET_PATH = "/home/aki/EDGE/data/validation_set"

In [12]:
embeddings = {}
for person in os.listdir(DATASET_PATH):
    person_path = os.path.join(DATASET_PATH, person)

    if not os.path.isdir(person_path):
        continue

    embeddings[person] = []

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            print(f"Failed to load: {img_path}")
            continue

        status, emb = get_face_embedding(img)

        if emb is not None:
            embeddings[person].append(emb)
        else:
            print(f"Skipping (no face): {img_path}")

print("Embedding extraction complete")

Skipping (no face): /home/aki/EDGE/data/validation_set/person_1/image.png
Skipping (no face): /home/aki/EDGE/data/validation_set/person_2/image copy 3.png
Embedding extraction complete


In [13]:
genuine_scores = []
imposter_scores = []

persons = list(embeddings.keys())

# Genuine (same person)
for person in persons:
    embs = embeddings[person]
    for i in range(len(embs)):
        for j in range(i + 1, len(embs)):
            score = calculate_similarity(embs[i], embs[j])
            genuine_scores.append(score)

# Imposter (different persons)
for i in range(len(persons)):
    for j in range(i + 1, len(persons)):
        embs1 = embeddings[persons[i]]
        embs2 = embeddings[persons[j]]

        for e1 in embs1:
            for e2 in embs2:
                score = calculate_similarity(e1, e2)
                imposter_scores.append(score)

print("Genuine samples:", len(genuine_scores))
print("Imposter samples:", len(imposter_scores))

Genuine samples: 30
Imposter samples: 75


In [14]:
print("Genuine → min:", min(genuine_scores), "max:", max(genuine_scores))
print("Imposter → min:", min(imposter_scores), "max:", max(imposter_scores))

Genuine → min: 0.39592885971069336 max: 1.0
Imposter → min: -0.10278294235467911 max: 0.12433857470750809


In [15]:
from ml.evaluation.metrics import evaluate_metrics
from backend.config.settings import ACCEPT_THRESHOLD, REJECT_THRESHOLD

results = evaluate_metrics(
    genuine_scores,
    imposter_scores,
    ACCEPT_THRESHOLD,
    REJECT_THRESHOLD
)

print(results)

{'FAR': 0.0, 'FRR': 0.0, 'accuracy': 1.0}


In [17]:
import importlib
import backend.utils.logger as logger

importlib.reload(logger)

<module 'backend.utils.logger' from '/home/aki/EDGE/backend/utils/logger.py'>

In [18]:
logger.log_metrics(f"Evaluation Results: {results}")
print("Metrics logged")

Metrics logged
